In [ ]:
import pandas as pd
from assign_regions import load_regions, nearest_stations, station_regions
from avalanche_euregio import get_avalanche_data
from data_engineering_project.combine import build_combined

In [ ]:
# load stations dataset
# automate this step?

df_stations = pd.read_csv("data/station_data.csv").drop(columns=["substation", "tb100"], errors="ignore")

df_metadata = pd.read_csv("data/stations_metadata.csv")[["id", "Stationsname", "Länge [°E]", "Breite [°N]", "Höhe [m]", "Bundesland" ]]
df_metadata.columns = ["station", "name", "lon", "lat", "elevation_m", "state"]
df = df_stations.merge(df_metadata, on="station", how="left", validate="many_to_one")

print("head")
print(df.head(10))
print("tail")
print(df.tail(10))

RENAME = {
    "time":  "timestamp",
    "tl":    "air_temp_c",          # air temp 2 m, °C
    "rr":    "precip_mm",           # precipitation (total hours ), mm
    "rrm":   "precip_minutes",      # precipitation duration, min
    "sh":    "snow_depth_cm",       # snow depth, cm
    "ff":    "wind_speed_ms",       # wind speed, m/s
    "ffx":   "wind_gust_ms",        # peak wind speed, m/s
    "dd":    "wind_dir_deg",        # wind direction , °
    "rf":    "rel_humidity_pct",    # relative humidty, %
    "p":     "pressure_hpa",        # air pressure, hPa
    "so_h":  "sunshine_h",          # sunshine duration, h
    "tb10":  "soil_temp_10cm_c",    # soil temperature −10 cm, °C
 }
df = df.rename(columns=RENAME)
df.info()
df.to_csv("data/station_data_cleaned.csv", index=False)

stations = (df.drop_duplicates("station")
              [["station", "name", "lon", "lat", "elevation_m"]])


avalanche_regions = load_regions("data/AT-07_micro-regions.geojson.json")

print(station_regions(stations, avalanche_regions))          # in which region is which station located
print(nearest_stations(stations, avalanche_regions, k=3))    # not relevant tbh

#avalanche_data = get_avalanche_data("01-10-2019", "01-05-2026") # load data
# avalanche_data.info()
# avalanche_data.tail(40)
# avalanche_data.to_csv("data/avalanche_bulletins.csv", index=False)


ad_df = pd.read_csv("data/avalanche_bulletins.csv")
ad_df.tail(12)

combined = build_combined(df, ad_df, avalanche_regions)
combined.to_csv("data/combined.csv", index=False)
combined.info()



In [4]:
combined.head(20)

,date,season,region,region_name,status,danger_max,danger_above,danger_below,elev_limit,danger_am,...,dist_snow_station_km,precip_sum_mm,dist_precip_station_km,snow_delta_24h_cm,snow_delta_72h_cm,wind_dir_deg,precip_sum_3d_mm,precip_sum_7d_mm,air_temp_mean_3d,wind_gust_max_3d
0,2019-11-16,2019/20,AT-07-01,Allgäuer Alpen,ok,2,2,1.0,treeline,2,...,12.8,5.977253,12.8,4.382609,18.181406,120.0,14.711813,22.566130,-7.050659,33.115210
1,2019-11-16,2019/20,AT-07-02,Östliche Lechtaler Alpen - Ammergauer Alpen,ok,2,2,1.0,treeline,2,...,31.5,19.630070,22.7,20.446380,39.474761,122.0,39.932324,47.783351,-5.481764,26.540503
2,2019-11-16,2019/20,AT-07-03,Mieminger Gebirge,ok,2,2,1.0,treeline,2,...,32.0,16.585057,20.9,10.360803,25.820254,164.0,42.236114,50.554827,-5.720043,27.587469
3,2019-11-16,2019/20,AT-07-04,Karwendel,ok,3,3,2.0,treeline,3,...,21.5,2.416858,6.5,-5.902464,17.294630,169.0,32.367873,47.708208,-4.710968,33.566196
4,2019-11-16,2019/20,AT-07-05,Brandenberger Alpen,ok,2,2,1.0,treeline,2,...,31.6,3.110328,30.5,-9.113960,16.827651,164.0,35.293271,53.898423,-3.049449,28.975875
5,2019-11-16,2019/20,AT-07-06,Wilder Kaiser - Waidringer Alpen,ok,2,2,1.0,treeline,2,...,60.3,3.185389,60.3,-6.662550,19.574828,153.0,35.471351,54.394115,-2.332753,27.385997
6,2019-11-16,2019/20,AT-07-07,Westliche Lechtaler Alpen,ok,2,2,1.0,treeline,2,...,0.0,6.151518,0.0,-11.013732,-2.356333,115.0,15.858704,24.897361,-7.026040,33.856725
7,2019-11-16,2019/20,AT-07-08,Zentrale Lechtaler Alpen,ok,2,2,1.0,treeline,2,...,11.6,9.626572,11.6,7.784674,22.580781,124.0,22.888911,31.816637,-6.653791,32.785487
8,2019-11-16,2019/20,AT-07-09,Grieskogelgruppe,ok,2,2,1.0,treeline,2,...,29.8,8.965354,15.7,3.457468,25.754279,169.0,34.189230,45.545099,-4.504623,28.600130
9,2019-11-16,2019/20,AT-07-10,Westliche Verwallgruppe,ok,2,2,2.0,NaN,2,...,1.3,6.134068,0.0,-8.550158,0.923286,100.0,13.124035,18.289195,-4.455374,26.602373
